# Hipótesis B: experiencia y confianza en la precisión de la IA

B) Los años de experiencia influyen en la confianza en la precisión de las herramientas de IA.

Esta hipótesis es **bivariada** (dos variables) y usa el **Subgrupo 2**. Seguimos el flujo de la práctica 6 de la cátedra: gráfico, normalidad, homocedasticidad, elección de la prueba y lectura con alfa = 0,05.

In [1]:
%pip install pandas seaborn matplotlib scipy wget

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from wget import download
from os import path, listdir
import pandas as pd
# PARA AUMENTAR EL ANCHO PERMITIDO Y Q NO VEA DATOS CORTADOS
pd.set_option('display.max_colwidth', None)
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import shapiro, kstest

In [ ]:
if not path.exists("schema.csv"):
  download("https://media.githubusercontent.com/media/StackExchange/Survey/refs/heads/main/packages/archive/2025/schema.csv")
if not path.exists("results.csv"):
  download("https://media.githubusercontent.com/media/StackExchange/Survey/6ae2209cd35a55e92739ba82f7887fb86807a069/packages/archive/2025/results.csv")

In [ ]:
# low_memory=False evita el DtypeWarning de las columnas con tipos mezclados
raw_dataset_results = pd.read_csv("results.csv", low_memory=False)
raw_dataset_schema = pd.read_csv("schema.csv")

In [ ]:
# 1. Definición de las listas de columnas para cada subgrupo
cols_g1 = ['Country', 'WorkExp', 'ConvertedCompYearly', 'CompTotal', 'Currency', 'Age', 'EdLevel', 'Employment', 'YearsCode']
cols_g2 = ['WorkExp', 'AIAcc', 'JobSat', 'AISent', 'AISelect', 'AIThreat', 'AILearnHow']

# Para el Grupo 3 extraemos dinámicamente todas las de JobSatPoints (del 1 al 15 + TEXT) para evitar hardcodearlas
cols_jobsat = [col for col in raw_dataset_results.columns if col.startswith('JobSatPoints')]
cols_g3 = ['NewRole', 'JobSat'] + cols_jobsat

cols_g4 = ['ConvertedCompYearly', 'WorkExp', 'OrgSize', 'AISelect', 'RemoteWork', 'DevType', 'Industry', 'ICorPM']

# 2. Creación de los DataFrames independientes para cada subgrupo
# Usamos .copy() para que cada compañero pueda limpiar nulos o transformar datos sin afectar el dataset original

# Subgrupo 1: Perfil Económico (Para Hipótesis A y HU1)
df_perfil_economico = raw_dataset_results[cols_g1].copy()

# Subgrupo 2: Impacto de la IA (Para Hipótesis B y HB1)
df_impacto_ia = raw_dataset_results[cols_g2].copy()

# Subgrupo 3: Cambio de Rol (Para Hipótesis C)
df_cambio_rol = raw_dataset_results[cols_g3].copy()

# Subgrupo 4: Premio Salarial (Para Hipótesis M4)
df_premio_salarial = raw_dataset_results[cols_g4].copy()

Control: las columnas de cada subgrupo existen en `results` (cada lista debe salir vacía) y el esquema describe las del subgrupo 2.

In [ ]:
for nombre, cols in {"g1": cols_g1, "g2": cols_g2, "g3": cols_g3, "g4": cols_g4}.items():
    print(nombre, [c for c in cols if c not in raw_dataset_results.columns])

raw_dataset_schema[raw_dataset_schema["qname"].isin(cols_g2)][["qid", "qname", "type", "question"]]

Transformacion sobre Subgrupo 2: Experiencia vs Confianza en la IA

In [ ]:
# 1. Valores imposibles de WorkExp (99 o más años) pasan a nulo. No se borra la fila.
print("WorkExp a nulo:", (df_impacto_ia["WorkExp"] >= 99).sum())
df_impacto_ia.loc[df_impacto_ia["WorkExp"] >= 99, "WorkExp"] = np.nan

Justificacion de limpieza para el informe:

- **Valores imposibles en `WorkExp`:** hay un pico aislado en 99 y 100 años de experiencia (incluso en personas de 18 a 24 años), lo que indica errores de carga o respuestas de relleno. Pasarlos a nulo evita que distorsionen el análisis. Se elige nulo y no eliminar la fila para conservar las respuestas de esa persona en las demás preguntas.
- **Rechazo a la imputación de datos:** `WorkExp` y `AIAcc` tienen nulos (la encuesta tiene preguntas condicionales y no sabemos si la persona no respondió o no le apareció la pregunta). Rellenarlos con promedios o medianas introduciría un sesgo artificial en una encuesta de opinión.
- **Estrategia de eliminación segura (Subsetting):** no se aplica un `.dropna()` global sobre las 172 columnas, porque las columnas de texto libre con altísima tasa de nulos eliminarían casi todas las filas válidas. Primero se aíslan las dos variables de interés y recién entonces se eliminan las filas con nulos solo en ellas.
- **Codificación manual de `AIAcc`:** es ordinal (5 niveles). El orden se define a mano con un diccionario porque pandas ordenaría alfabéticamente.

Validación de Hipótesis B

Los años de experiencia profesional están relacionados con el nivel de confianza en la precisión de las herramientas de IA?

Partimos del Subgrupo 2 (Impacto de la IA), cuyas columnas ya fueron aisladas. Las variables que usamos son:

- **Variable 1 (`WorkExp`):** numérica (años de experiencia profesional). Está sesgada hacia valores altos.
- **Variable 2 (`AIAcc`):** categórica ordinal (MC) de 5 niveles, de "Highly distrust" (1) a "Highly trust" (5).

Exclusión de variables no necesarias: las demás columnas del subgrupo (`JobSat`, `AISent`, `AISelect`, `AIThreat`, `AILearnHow`) no intervienen en esta hipótesis.

Planteo estadístico: **H0:** la experiencia es la misma en los 5 niveles de confianza. **H1:** en al menos un nivel es distinta. Nivel de significación: alfa = 0,05.

In [ ]:
# 1. Codificación ordinal de AIAcc (1 = desconfía mucho, 5 = confía mucho)
orden_aiacc = {"Highly distrust": 1, "Somewhat distrust": 2, "Neither trust nor distrust": 3,
               "Somewhat trust": 4, "Highly trust": 5}
df_impacto_ia["AIAcc_num"] = df_impacto_ia["AIAcc"].map(orden_aiacc)

# Control: respuestas no nulas que quedaron sin código (debe salir vacío)
print(df_impacto_ia[df_impacto_ia["AIAcc"].notna() & df_impacto_ia["AIAcc_num"].isna()]["AIAcc"].value_counts())

# 2. Nos quedamos con las dos variables y con las filas que tienen ambas respuestas
df_hipotesis_b = df_impacto_ia[["WorkExp", "AIAcc", "AIAcc_num"]].dropna().copy()

print(f"Total de observaciones listas para el análisis: {len(df_hipotesis_b)}")
df_hipotesis_b.head()

Análisis de las variables

Antes de aplicar las pruebas miramos cómo se distribuye cada variable. `AIAcc` es una variable cualitativa ordinal: tiene un orden, pero **no sabemos que la distancia entre niveles sea la misma**, por eso no corresponde calcular medias de ella. `WorkExp` se revisa con su asimetría (`skew`).

In [ ]:
print("Personas por nivel de AIAcc:")
print(df_hipotesis_b["AIAcc_num"].value_counts().sort_index())

print("Asimetría (skew) de WorkExp:", round(df_hipotesis_b["WorkExp"].skew(), 2))
print("Media:", round(df_hipotesis_b["WorkExp"].mean(), 2), "| Mediana:", df_hipotesis_b["WorkExp"].median())

sns.histplot(df_hipotesis_b["WorkExp"], bins=40)
plt.title("Distribución de WorkExp")
plt.show()

Paso 1: Exploración visual

Graficamos la experiencia de cada nivel de confianza con un boxplot con muesca (`notch=True`). La caja es el 50% central, la línea es la mediana y la muesca es una zona de incertidumbre de la mediana: si las muescas de dos cajas no se superponen, las medianas probablemente difieren.

In [ ]:
print("Mediana de WorkExp por nivel de AIAcc:")
print(df_hipotesis_b.groupby("AIAcc_num")["WorkExp"].median())

plt.figure(figsize=(8, 4.5))
sns.boxplot(x="AIAcc_num", y="WorkExp", data=df_hipotesis_b, notch=True)
plt.xlabel("AIAcc (1 = desconfía mucho, 5 = confía mucho)")
plt.ylabel("Años de experiencia (WorkExp)")
plt.title("Experiencia según confianza en la precisión de la IA")
plt.show()

Paso 2: Normalidad

Para elegir entre una prueba paramétrica (ANOVA) y una no paramétrica hay que ver si `WorkExp` es normal **dentro de cada grupo**. Usamos Shapiro-Wilk, Kolmogorov-Smirnov (estandarizando antes los datos) y QQ plot. La H0 de normalidad dice que los datos siguen una distribución normal: si p < 0,05 se rechaza. Con grupos de más de 5.000 datos `scipy` avisa que el p-valor de Shapiro es aproximado, es esperable.

In [ ]:
grupos_b = [g["WorkExp"] for _, g in df_hipotesis_b.groupby("AIAcc_num")]

for i, g in enumerate(grupos_b, start=1):
    stat, p = shapiro(g)
    print(f"AIAcc = {i}: Shapiro-Wilk Estadístico={stat:.3f}, p-valor={p:.3g}")

for i, g in enumerate(grupos_b, start=1):
    stat, p = kstest((g - g.mean()) / g.std(), 'norm')
    print(f"AIAcc = {i}: Kolmogorov-Smirnov Estadístico={stat:.3f}, p-valor={p:.3g}")

In [ ]:
fig, ejes = plt.subplots(1, 5, figsize=(18, 3.5))
for i, g in enumerate(grupos_b):
    stats.probplot(g, dist="norm", plot=ejes[i])
    ejes[i].set_title(f"QQ plot AIAcc = {i + 1}")
plt.tight_layout()
plt.show()

Si los puntos del QQ plot siguen la recta, los datos son normales; si se curvan, no lo son. Si los tests rechazan normalidad y el QQ plot se curva, **no se puede usar ANOVA** y corresponde una prueba no paramétrica.

Paso 3: Homocedasticidad (Levene)

H0: las varianzas de los 5 grupos son **iguales**. Si p < 0,05 se rechaza y las varianzas son distintas.

In [ ]:
stat, p = stats.levene(*grupos_b)
print(f"Test de Levene para WorkExp: Estadístico={stat:.3f}, p-valor={p:.3g}")

alpha = 0.05
if p > alpha:
    print("No hay suficiente evidencia para rechazar la hipótesis nula.")
    print("Las varianzas de WorkExp son iguales entre los grupos de AIAcc.")
else:
    print("Se rechaza la hipótesis nula.")
    print("Las varianzas de WorkExp son diferentes entre los grupos de AIAcc.")

Paso 4: Prueba de hipótesis (Kruskal-Wallis)

Hay **5 grupos independientes** (cada persona está en un solo nivel de confianza) y `WorkExp` no es normal. Por eso usamos **Kruskal-Wallis**, la alternativa no paramétrica al ANOVA para comparar más de dos grupos. Solo dice **si hay diferencia** en algún grupo, no entre cuáles ni en qué sentido.

In [ ]:
stat_kw, p_kw = stats.kruskal(*grupos_b)
print(f"Test de Kruskal-Wallis para WorkExp: Estadístico={stat_kw:.3f}, p-valor={p_kw:.3g}")

alpha = 0.05
if p_kw > alpha:
    print("No hay suficiente evidencia para rechazar la hipótesis nula.")
    print("No hay una diferencia significativa en WorkExp entre los niveles de AIAcc.")
else:
    print("Se rechaza la hipótesis nula.")
    print("Existe una diferencia significativa en WorkExp entre los niveles de AIAcc.")

Paso 5: Sentido de la asociación

Kruskal-Wallis indica que hay diferencia, pero no en qué sentido. Para eso calculamos la correlación de Spearman entre WorkExp y AIAcc_num. 
Esta correlación trabaja con rangos (el orden de los valores), por lo que sirve para una variable ordinal y no exige normalidad. 
Positiva significa que a más experiencia, más confianza; negativa, que a más experiencia, menos confianza; cercana a 0, que no hay asociación.

In [ ]:
tabla_corr = df_hipotesis_b[["WorkExp", "AIAcc_num"]].corr(method="spearman")
rho = tabla_corr.loc["WorkExp", "AIAcc_num"]
print(tabla_corr)
print(f"\nCorrelación de Spearman = {rho:.3f}")

Como alfa es = 0,05 se rechaza H0.
La experiencia profesional difiere de forma significativa entre los niveles de confianza en la precisión de la IA (H = 38,68; p = 8,1e-8). La correlación de Spearman es negativa (rho = -0,034): a mayor confianza en la IA le corresponde una experiencia levemente menor.